In [1]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime

print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import (
    TOKENIZER_SAVE_DIRECTORY,
    OUR_DATASET_DIRECTORY,
    OUR_DATASET_DIRECTORY_2,
    OUR_DATASET_DIRECTORY_3,
)
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, Verbose
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

added '/autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=13, micro=12, releaselevel='final', serial=0)


In [2]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [3]:
import importlib
import LLM.model

_ = importlib.reload(LLM.model)
from LLM.model import Model

In [4]:
try:
    torch.cuda.empty_cache()
    del model  # type: ignore
    torch.cuda.empty_cache()
except Exception:
    pass
model = Model(
    save_name="models_1.6_tests",
    tokenizer=TOKENIZER_SAVE_DIRECTORY.joinpath("our_tokenizer"),
    device="cuda",
    depth=6,
    head_dim=128,
    context_size=4096 * 1,
    nb_heads_mult=5.0,
)
model.show_infos()

loading the tokenizer from: /autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation/tokenizer_save/our_tokenizer.json
Padding vocab_size from 585 to 640 for efficiency
Scaling the LR for the AdamW parameters ∝1/√(384/768) = 1.414214
GPTConfig(sequence_len=8192, vocab_size=585, n_layer=6, n_head=3, n_kv_head=3, n_embd=384, window_pattern='SSSL')
11_845_932 total params (embeding: 983_040 | last layer: 245_760 | transformer: 10_617_120)
on device: device(type='cuda', index=0), with effective context_size: 4096


In [5]:
ID_1, _ = model.save("un_trained")

saving the tokenizer to: /autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation/models_save/models_1.6_tests/tokenizer.json
saving the historique to: /autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation/models_save/models_1.6_tests/versions/version_0002_un_trained/history.json


In [6]:
try:
    if "dataset" in globals():
        raise FileExistsError
    dataset = svg_dataset.SVGDataset(
        OUR_DATASET_DIRECTORY,
        context_size=model.context_size,
        tokenizer=model.tokenizer.encode,
        decoder=model.tokenizer.decode,
    )
except FileExistsError:
    pass

In [7]:
model.train(
    dataset=dataset,
    batch_size=16,
    nbEpoches=1,
    timeLimite=999_999,
    verbose=Verbose.liveProgress,
)


starting epoch: 1


W0316 10:40:29.662000 261993 .venv/lib/python3.13/site-packages/torch/_inductor/utils.py:1613] [0/0] Not enough SMs to use max_autotune_gemm mode


finished batches in 8 min 29.1 sec                   
saving the tokenizer to: /autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation/models_save/models_1.6_tests/tokenizer.json
saving the historique to: /autofs/unitytravail/travail/janglichau/PFE/PFE_LLM_art_generation/models_save/models_1.6_tests/versions/version_0003_checkpoint-1/history.json
finished epoches in 8 min 29.6 sec
trained on: 863 batch (13793 chuncks) in 8 min 29.1 sec
 -> 1.70 batch/sec | 27.09 chuncks/sec
 -> CE: 0.9705 | PPL: 2.639 | top-1: 76.29%
